In [0]:
%sql
SET TAG ON TABLE bookstoreprof.silver.customers_silver project = bookstore;
SET TAG ON TABLE bookstoreprof.silver.customers_silver contains_pii;


In [0]:
%sql
ALTER TABLE bookstoreprof.silver.customers_silver
SET TAGS ('department' = "sales", "compliance" = 'GDPR')

In [0]:
SELECT * FROM system.billing.usage

In [0]:
SELECT 
  date_trunc('day', usage_start_time) AS usage_day,
  identity_metadata.run_as AS user_email,
  sku_name AS compute_type,
  SUM(usage_quantity) AS total_dbus
FROM system.billing.usage
WHERE usage_unit = 'DBU'
GROUP BY user_email, sku_name, usage_day
ORDER BY total_dbus DESC

-- You can create a dashbord for this cost monitoring and share with the responsible person

In [0]:
%sql
DESC TABLE bookstoreprof.silver.customers_silver

In [0]:
USE CATALOG bookstoreprof;
CREATE OR REPLACE VIEW silver.customers_vw AS 
SELECT 
  customer_id,
  CASE 
    WHEN is_member("hr_team") THEN email
    ELSE 'RADICATED'
  END AS email,
  gender,
  CASE 
    WHEN is_member("hr_team") THEN first_name
    ELSE 'RADICATED'
  END AS first_name,
  CASE 
    WHEN is_member("hr_team") THEN last_name
    ELSE 'RADICATED'
  END AS last_name,
  CASE
    WHEN is_member("hr_team") THEN street
    ELSE 'RADICATED'
  END AS stree,
  city,
  country  
FROM silver.customers_silver;

In [0]:
select * from silver.customers_vw

In [0]:
CREATE OR REPLACE TABLE silver.customers_fr_vw AS 
SELECT 
  *
FROM silver.customers_vw
WHERE  
  CASE 
    WHEN is_member("hr_team") THEN TRUE
    ELSE country = 'France'
  END 

In [0]:
select * from silver.customers_fr_vw

Row Filters and Column Masks
In the last lecture, we covered dynamic views as a solution to share filtered data. However, this approach required creating additional objects within the schema. Unity Catalog now simplifies this process by supporting row filters and column masks directly on the table itself. So, when querying the table, users automatically see only the rows and columns they are authorized to access.



Column masks

To dynamically mask a column in a table, start by defining the masking logic in a user-defined function. For example:




CREATE FUNCTION email_mask(email STRING)
RETURN CASE WHEN is_member('admins_demo') THEN email ELSE 'REDACTED' END;


Now, you can apply this function on the column in your table:

ALTER TABLE customers_silver ALTER COLUMN email SET MASK email_mask;



Row filters

To dynamically filter rows in a table, start by defining the filtering logic in a user-defined function. For example:



CREATE FUNCTION fr_filter(country STRING)
RETURN IF(is_member('admins_demo'), true, country="France");


Now, you can apply this function as a row filter on the table:

ALTER TABLE customers_silver SET ROW FILTER fr_filter ON (country);



Note that the is_member function tests group membership at the workspace level. To test group membership at the account level in Unity Catalog, use the is_account_group_member function instead.

In [0]:
CREATE FUNCTION email_mask(email STRING) RETURNS STRING
RETURN 
  CASE 
    WHEN is_member("hr_team") THEN email
    ELSE 'RADICATED'
  END

In [0]:
ALTER TABLE bookstoreprof.silver.customers_silver ALTER COLUMN email SET MASK email_mask;

In [0]:
select * from bookstoreprof.silver.customers_silver limit 10

In [0]:
-- To dynamically filter rows in a table, start by defining the filtering logic in a user-defined function.
CREATE OR REPLACE FUNCTION fr_filter(country STRING)
RETURN IF(
  is_member("hr_team"),
  TRUE,
  country = 'France'
)

In [0]:
ALTER TABLE bookstoreprof.silver.customers_silver SET ROW FILTER fr_filter ON (country);

In [0]:
select * from bookstoreprof.silver.customers_silver limit 10